# Covariance Estimation, Shrinkage y Portfolio Robustness

> Esta semana vamos a profundizar en un problema directamente relacionado con el estudio de Portfolio Construction de Anton Vorobets, los modelos de factores de QuantFrame y el Quant Desk:

---
$  \boxed{\text{Una matriz de covarianza estimada no es la verdadera matriz de riesgo.}} $
Hasta ahora hemos trabajado con matrices $ \Sigma $  conocidas. 

En un problema real, tenemos que estimarlas a partir de datos históricos.

Esto introduce un riesgo importante: un optimizador puede explotar errores de estimación y construir un portafolio aparentemente óptimo, pero muy frágil fuera de muestra.

--- 

Hoy estudiaremos ese problema desde tres perspectivas: matemática, computacional y práctica.

| Día | Tema | Competencia |
|---|---|---|
| **Lunes 12** | Covariance estimation y shrinkage | Matemáticas + estadística + portfolio |
| Martes 13 | Factor risk vs idiosyncratic risk | Modelos factoriales + Python |
| Miércoles 14 | Order book, spread y adverse selection | Mercados + microestructura |
| Jueves 15 | Entropy Pooling y views inconsistentes | Probabilidad + portfolio/risk |
| Viernes 16 | Lectura crítica de research: validación de estrategias | Investigación cuantitativa |
| Sábado 17 | Quant Desk: integración y decisión de riesgos | Aplicación práctica |
| Domingo 18 | **Descanso** | Sin actividades |

<b> Estadística |¿Puede Shrinkage superar a un optimizador aparentemente más preciso? </b>

Vamos a simular ocho activos con una fuente común de riesgo y componentes idiosincráticos.

```python
import numpy as np

rng = np.random.default_rng(117)
n_assets = 8
T_train = 50
T_test = 252
T = T_train + T_test

common_factor = rng.normal(0, 0.007, size=(T, 1))
idio_noise = rng.normal(0, 0.009, size=(T, n_assets))
returns = common_factor + idio_noise

train = returns[:T_train]
test = returns[T_train:]
```

La muestra de entrenamiento es deliberadamente pequeña.

**Parte A — Covarianza muestral**

Calcula:

$\hat{\Sigma}_{sample} = \text{Cov}(R_{train})$.

Después construye una matriz regularizada:

$\hat{\Sigma}_{shrink} = (1 - \lambda)\hat{\Sigma}_{sample} + \lambda D$

donde:

$D = \text{diag}(\hat{\Sigma}_{sample})$

y:

$\lambda = 0.50.$

Este shrinkage reduce las covarianzas fuera de la diagonal sin modificar las varianzas individuales.

No es necesario estimar un $\lambda$ óptimo. Hoy lo fijamos antes de ver Test.

**Parte B — Portfolio Optimization**

Construye tres portafolios:

**Portfolio A: Sample Minimum Variance**

$w_A = \frac{\hat{\Sigma}_{sample}^{-1} \mathbf{1}}{\mathbf{1}^T \hat{\Sigma}_{sample}^{-1} \mathbf{1}}.$

**Portfolio B: Shrinkage Minimum Variance**

$ w_B = \frac{\hat{\Sigma}_{shrink}^{-1} \mathbf{1}}{\mathbf{1}^T \hat{\Sigma}_{shrink}^{-1} \mathbf{1}} $

**Portfolio C: Equal Weight**

$ w_C = [1/8 \quad 1/8 \quad \dots \quad 1/8]^T $

Utiliza `np.linalg.solve()` en lugar de calcular explícitamente las matrices inversas.

**Parte C — Evaluación OOS**

Evalúa los tres portafolios exclusivamente sobre `test`.

Calcula:

$\sigma_{OOS} = \text{Std}(R_{test}w)\sqrt{252}.$

También calcula:

$\text{GrossExposure} = \sum_{i} |w_i|.$

Y:

$\text{MaxWeight} = \max_{i} |w_i|.$

| Portfolio | OOS Vol | Gross Exposure | Max Absolute Weight |
|---|---:|---:|---:|
| Sample MinVar | | | |
| Shrinkage MinVar | | | |
| Equal Weight | | | |


Construye:

| Portfolio | OOS Vol | Gross Exposure | Max Absolute Weight |
|---|---:|---:|---:|
| Sample MinVar | | | |
| Shrinkage MinVar | | | |
| Equal Weight | | | |

Preguntas:

¿Por qué la matriz muestral puede generar posiciones extremas aunque sus estimaciones sean estadísticamente razonables?

¿Por qué shrinkage introduce sesgo deliberadamente?

¿Bajo qué condiciones puede compensarse ese sesgo mediante menor error fuera de muestra?

**Advertencia importante:** La fórmula utilizada permite posiciones cortas. Si obtienes pesos negativos, no debes interpretar automáticamente el portfolio como ejecutable en un entorno long-only.

Además, una sola simulación no demuestra que shrinkage sea superior. Si te quedan cinco minutos, repite el experimento con 20 semillas y compara la mediana de volatilidad OOS.

**Entregable:** Un DataFrame y tres conclusiones breves.


In [ ]:
import numpy as np

rng = np.random.default_rng(117)
n_assets = 8
T_train = 50
T_test = 252
T = T_train + T_test

common_factor = rng.normal(0, 0.007, size=(T, 1))
idio_noise = rng.normal(0, 0.009, size=(T, n_assets))
returns = common_factor + idio_noise

train = returns[:T_train]
test = returns[T_train:]
# PARTE A - Covarianza Muestral

# calculando la matriz de cov 
cov_sample = np.cov(train, rowvar = False)
# matriz regularizada shrink
lambda_ = 0.50 
D = np.diag(cov_sample)
cov_shrink = (1 - lambda_)*cov_sample + (lambda_*D)

# PARTE B - optimización de portafolio 

# Portafolio A - Sample Minumum variance

